# Lesson 6.7: The Paired t-Test

Test whether a drug lowers blood pressure in the lesson's 8 patients measured before and after, with a paired t-test.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

## The question

A clinic measures the blood pressure of 8 patients before and after they start a new drug. Every patient's reading went down, yet the patients differ a lot from one another (readings from 129 to 161). Is the average drop of about 6 mmHg real evidence that blood pressure changed, or could it be chance?

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## Step 1: Enter the paired data

H0: the average change in blood pressure is 0. H1: the average change is not 0. Each patient appears in both lists in the same order, so each 'after' belongs to one 'before'. Each patient's difference is their own change.

In [1]:
import numpy as np  # Load NumPy for arrays and arithmetic, under the short name np.
from scipy import stats  # Load SciPy's statistics tools; the paired t-test lives in stats.

before = np.array([142, 138, 155, 129, 147, 161, 133, 145])  # Store the 8 'before' readings. The order matters: patient 1 is first in both lists.
after = np.array([135, 134, 148, 126, 139, 152, 130, 137])  # Store the 8 'after' readings in the same patient order, so the pairs line up.
diff = before - after  # Subtract to get each patient's own change. The paired test works on these differences, not on the two lists.

print('differences:', diff)  # Print the 8 differences to check that they all look sensible.
print(f'mean difference = {diff.mean():.3f}')  # Print the average drop, the centre of the differences.
print(f'sd of differences = {diff.std(ddof=1):.3f}')  # Print the SD of the differences (ddof=1): how much the drops vary from patient to patient.

differences: [7 4 7 3 8 9 3 8]
mean difference = 6.125
sd of differences = 2.416


**What this means.** Every difference is positive (7, 4, 7, 3, 8, 9, 3, 8), so all eight patients improved. The average drop is 6.125 mmHg and the drops vary by only 2.416. The drops are steady even though the patients' blood pressures differ widely. That steadiness is what the paired test uses.

## Step 2: Run the paired t-test

ttest_rel compares before and after pair by pair. A positive t means the 'before' values were higher. We also add a 95% confidence interval for the average drop.

In [2]:
paired = stats.ttest_rel(before, after)  # ttest_rel is the paired t-test: it tests whether the mean of before-minus-after differences is 0.
ci = paired.confidence_interval(confidence_level=0.95)  # Ask the result for a 95% confidence interval for the average drop.

print(f't  = {paired.statistic:.3f}')  # Print t, the average drop measured in standard errors. Positive means 'before' was higher.
print(f'df = {paired.df}')  # Print the degrees of freedom: 8 pairs minus 1 = 7.
print(f'p  = {paired.pvalue:.4f}')  # Print the two-sided p-value to 4 decimals.
print(f'95% CI for the drop: ({ci.low:.2f}, {ci.high:.2f})')  # Print the interval for the true average drop, rounded to 2 decimals.

t  = 7.169
df = 7
p  = 0.0002
95% CI for the drop: (4.10, 8.15)


**What this means.** t = 7.169 with df = 7 and p of about 0.0002. If the drug changed nothing, an average change this large would almost never occur, so we reject H0. The 95% interval says the true average drop is plausibly 4.10 to 8.15 mmHg. The p-value alone would not tell us how large that is.

## Step 3: Same result as a one-sample test

A paired t-test is just a one-sample t-test on the differences, testing whether their mean is 0. The numbers come out exactly the same.

In [3]:
one_sample = stats.ttest_1samp(diff, popmean=0)  # Run an ordinary one-sample t-test on the differences, testing whether their mean is 0.

print(f't  = {one_sample.statistic:.3f}')  # Print t from this version of the test.
print(f'p  = {one_sample.pvalue:.4f}')  # Print p from this version of the test.
print('same t as the paired test?', np.isclose(one_sample.statistic, paired.statistic))  # Check the two t values match. np.isclose compares numbers while ignoring tiny rounding noise.

t  = 7.169
p  = 0.0002
same t as the paired test? True


**What this means.** The one-sample test on the differences gives t = 7.169, the same as the paired test, and the check prints True. A paired t-test is simply a one-sample t-test on the differences, which is why paired designs behave like single-sample problems.

## Step 4: What the wrong test says

If we ignore the pairing and treat the 16 numbers as two independent groups, the big differences between patients swamp the 6 mmHg drop. The result is far less convincing.

In [4]:
wrong = stats.ttest_ind(before, after, equal_var=False)  # Deliberately use the WRONG test: treat before and after as two independent groups of 8 people.

print(f'independent test: t = {wrong.statistic:.2f}, p = {wrong.pvalue:.3f}')  # Print the independent test's t and p, to see what ignoring the pairing does.
print(f'paired test:      t = {paired.statistic:.2f}, p = {paired.pvalue:.4f}')  # Print the correct paired result next to it for comparison.

independent test: t = 1.25, p = 0.231
paired test:      t = 7.17, p = 0.0002


**What this means.** Treating the readings as independent gives t = 1.25 and p = 0.231, which is not significant. The paired test gives t = 7.17 and p = 0.0002. Same numbers, opposite conclusions. Differences between patients swamp the 6 mmHg drop unless the pairing is used. Choose the test from how the data were collected.

## So, what is the answer?

On average, blood pressure fell by 6.1 mmHg after the drug (paired t = 7.169, df = 7, p = 0.0002; 95% CI 4.10 to 8.15), and every patient improved. Caution: with no control group we cannot say the drug caused the drop (rest, time or being measured may also matter), and eight patients is a small sample.

## Now you try

A company runs a step challenge. It records the average daily steps of 10 employees in the month before the challenge and in the month during it. Did the challenge change average steps? Use a paired t-test, and compare with what the wrong (independent) test would say.

**Data:** before = [9024, 7900, 9000, 10157, 6986, 8304, 7305, 8190, 6219, 7131]
after = [9445, 8759, 9958, 10128, 7168, 9063, 7008, 8505, 6680, 8133]

1. Compute each employee's change (after minus before), then the mean and SD of the changes.
2. Run the paired t-test and report t, df and p.
3. Give the 95% confidence interval for the average change.
4. Run the independent (Welch) test on the same numbers and explain why it disagrees.

*Hint:* Use stats.ttest_rel(after, before) for the paired test and .confidence_interval(0.95). The independent version is stats.ttest_ind(after, before, equal_var=False).

In [ ]:
# Your turn: write your code here


### Solution (try first, then peek)

In [ ]:
import numpy as np
from scipy import stats

before = np.array([9024, 7900, 9000, 10157, 6986, 8304, 7305, 8190, 6219, 7131])
after = np.array([9445, 8759, 9958, 10128, 7168, 9063, 7008, 8505, 6680, 8133])
gain = after - before
res = stats.ttest_rel(after, before)
ci = res.confidence_interval(0.95)
print('changes:', gain)
print(f'mean change = {gain.mean():.1f}, sd = {gain.std(ddof=1):.1f}')
print(f'paired t = {res.statistic:.3f}, df = {res.df}, p = {res.pvalue:.4f}')
print(f'95% CI: ({ci.low:.0f}, {ci.high:.0f})')
ind = stats.ttest_ind(after, before, equal_var=False)
print(f'wrong independent test: t = {ind.statistic:.2f}, p = {ind.pvalue:.3f}')

changes: [ 421  859  958  -29  182  759 -297  315  461 1002]
mean change = 463.1, sd = 434.9
paired t = 3.367, df = 9, p = 0.0083
95% CI: (152, 774)
wrong independent test: t = 0.87, p = 0.398


**Answer.** Steps rose by 463.1 on average (SD of changes 434.9). The paired test gives t = 3.367, df = 9, p = 0.0083, and the 95% interval is 152 to 774 extra steps a day, so we reject H0. The independent test gives t = 0.87, p = 0.398 because the large differences between people hide the gain. Caution: with no control group we cannot say the challenge caused the rise.

## The same question in R, SPSS and Stata

**R**

```r
before <- c(142, 138, 155, 129, 147, 161, 133, 145)  # Store the 'before' readings for the 8 patients.
after  <- c(135, 134, 148, 126, 139, 152, 130, 137)  # Store the 'after' readings in the same patient order, so pairs match.

t.test(before, after, paired = TRUE)  # paired = TRUE makes t.test work on before minus after pair by pair. It prints t, df, p and a 95% interval.
t.test(before - after, mu = 0)    # same thing: one-sample test on the differences  # The same test as a one-sample test on the differences against 0; it gives the same numbers, which shows what 'paired' means.
```

How to read it: Look at t = 7.1692, df = 7, p-value = 0.0001823: far below 0.05, so reject H0. The 95 percent confidence interval, 4.105 to 8.145, is the plausible size of the average drop, and the mean difference is 6.125. The second call repeats the same numbers.

**SPSS** (Analyze > Compare Means > Paired-Samples T Test)

```
DATA LIST FREE / before after.
BEGIN DATA
142 135 138 134 155 148 129 126
147 139 161 152 133 130 145 137
END DATA.
T-TEST PAIRS=before WITH after (PAIRED)
  /CRITERIA=CI(.95).
```

How to read it: In the Paired Samples Test table the Mean is 6.125 (before minus after). Read t = 7.169, df = 7 and Sig. (2-tailed) = 0.000 (really 0.0002). The 95% interval 4.105 to 8.145 is for the average drop. We reject H0.

**Stata**

```
clear
input before after
142 135
138 134
155 148
129 126
147 139
161 152
133 130
145 137
end
ttest before == after
```

How to read it: In the output look at the diff row: mean 6.125 with its 95% interval (4.105 to 8.145). Then t = 7.1692 with 7 degrees of freedom and the middle line Pr(|T| > |t|) = 0.0002, the two-sided p-value. Below 0.05, so reject H0.